# Lesson 4:  Pattern Mining

## Objective
Use Apriori and FP-Growth algorithms to discover Association Rules in a transactional dataset.


## Frequent Pattern using Apriori Algorithm

### Apriori function :
- is an algorithm for frequent item set mining and association rule learning over relational databases.
- It proceeds by identifying the frequent individual items in the database and extending them to larger and larger item sets as long as those item sets appear sufficiently often in the database.

# Generating Frequent Itemsets

The apriori function expects data in a `one-hot` encoded pandas DataFrame. Suppose we have the following transaction data:


### Step 1: Simulating Transactional Data (Market Basket)
We create a dataset of shopping baskets.


In [4]:
dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

- We can transform it into the right format via the TransactionEncoder as follows:

In [6]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder


In [36]:
te = TransactionEncoder()
# The fit method is to learn what is the item to transaction
# the transform method is to Transform transactions into a one-hot encoded NumPy array.
te_array = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_array, columns=te.columns_)
df

,Apple,Corn,Dill,Eggs,Ice cream,Kidney Beans,Milk,Nutmeg,Onion,Unicorn,Yogurt
0,False,False,False,True,False,True,True,True,True,False,True
1,False,False,True,True,False,True,False,True,True,False,True
2,True,False,False,True,False,True,True,False,False,False,False
3,False,True,False,False,False,True,True,False,False,True,True
4,False,True,False,True,True,True,False,False,True,False,False


- Imagine that we need the min-support for transactions that fit 60%
- apriori function Get frequent itemsets from a one-hot DataFrame


### Step 2: Running Apriori Algorithm
Find frequent itemsets with a minimum support threshold.


In [37]:
from mlxtend.frequent_patterns import apriori

# Get frequent itemsets with min_support = 0.6
frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
print("--- Frequent Itemsets (Apriori) ---")
print(frequent_itemsets)


--- Frequent Itemsets (Apriori) ---
    support                     itemsets
0       0.8                       (Eggs)
1       1.0               (Kidney Beans)
2       0.6                       (Milk)
3       0.6                      (Onion)
4       0.6                     (Yogurt)
5       0.8         (Kidney Beans, Eggs)
6       0.6                (Eggs, Onion)
7       0.6         (Kidney Beans, Milk)
8       0.6        (Kidney Beans, Onion)
9       0.6       (Yogurt, Kidney Beans)
10      0.6  (Eggs, Onion, Kidney Beans)


  - `Eggs` is frequently 80% repeated in all transactions and `Beans` is the Most frequent item with 100%

### Step 3: Generating Association Rules

`Rule generation` is a common task in the mining of frequent patterns.
-  An association rule is an implication expression of the form X → Y, where X and Y are disjoint itemsets

From our theory class, we are clear what the support and confidence and lift are for the itemsets and rule.

For revision 'lift':
$$lift(A→C)={{confidence(A→C)} \over {support(C)}},range: [0,∞]$$

- The lift metric is commonly used to measure how much more often the antecedent and consequent of a rule A->C occur together than we would expect if they were statistically independent.
- If A and C are independent, the Lift score will be exactly 1.

In [38]:
from mlxtend.frequent_patterns import association_rules

# Generate rules with min_confidence = 0.7
rules = association_rules(frequent_itemsets,metric="confidence", min_threshold=0.7)
print(rules)


              antecedents            consequents  antecedent support  \
0          (Kidney Beans)                 (Eggs)                 1.0   
1                  (Eggs)         (Kidney Beans)                 0.8   
2                 (Onion)                 (Eggs)                 0.6   
3                  (Eggs)                (Onion)                 0.8   
4                  (Milk)         (Kidney Beans)                 0.6   
5                 (Onion)         (Kidney Beans)                 0.6   
6                (Yogurt)         (Kidney Beans)                 0.6   
7   (Kidney Beans, Onion)                 (Eggs)                 0.6   
8    (Kidney Beans, Eggs)                (Onion)                 0.8   
9           (Eggs, Onion)         (Kidney Beans)                 0.6   
10                (Onion)   (Kidney Beans, Eggs)                 0.6   
11                 (Eggs)  (Kidney Beans, Onion)                 0.8   

    consequent support  support  confidence  lift  representati

/opt/homebrew/Caskroom/miniconda/base/envs/thesis/lib/python3.11/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [39]:
print("--- Association Rules ---")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

--- Association Rules ---
              antecedents            consequents  support  confidence  lift
0          (Kidney Beans)                 (Eggs)      0.8        0.80  1.00
1                  (Eggs)         (Kidney Beans)      0.8        1.00  1.00
2                 (Onion)                 (Eggs)      0.6        1.00  1.25
3                  (Eggs)                (Onion)      0.6        0.75  1.25
4                  (Milk)         (Kidney Beans)      0.6        1.00  1.00
5                 (Onion)         (Kidney Beans)      0.6        1.00  1.00
6                (Yogurt)         (Kidney Beans)      0.6        1.00  1.00
7   (Kidney Beans, Onion)                 (Eggs)      0.6        1.00  1.25
8    (Kidney Beans, Eggs)                (Onion)      0.6        0.75  1.25
9           (Eggs, Onion)         (Kidney Beans)      0.6        1.00  1.00
10                (Onion)   (Kidney Beans, Eggs)      0.6        1.00  1.25
11                 (Eggs)  (Kidney Beans, Onion)      0.6     

- I Wouldn't like to manipulate the frozen sets so we make a list
- The Association Rules can be defined as : $$antecedent -> consequent = [ support = # , confidence = #]

In [40]:
for i , row in rules.iterrows():
    print(f"{i+1} - {list(row.antecedents)} -> {list(row.consequents)} = [support = {row.support} , confidence = {row.confidence}, lift ={row.lift}] , dependent = {row.lift != 1.0}" )



1 - ['Kidney Beans'] -> ['Eggs'] = [support = 0.8 , confidence = 0.8, lift =1.0] , dependent = False
2 - ['Eggs'] -> ['Kidney Beans'] = [support = 0.8 , confidence = 1.0, lift =1.0] , dependent = False
3 - ['Onion'] -> ['Eggs'] = [support = 0.6 , confidence = 1.0, lift =1.25] , dependent = True
4 - ['Eggs'] -> ['Onion'] = [support = 0.6 , confidence = 0.7499999999999999, lift =1.2499999999999998] , dependent = True
5 - ['Milk'] -> ['Kidney Beans'] = [support = 0.6 , confidence = 1.0, lift =1.0] , dependent = False
6 - ['Onion'] -> ['Kidney Beans'] = [support = 0.6 , confidence = 1.0, lift =1.0] , dependent = False
7 - ['Yogurt'] -> ['Kidney Beans'] = [support = 0.6 , confidence = 1.0, lift =1.0] , dependent = False
8 - ['Kidney Beans', 'Onion'] -> ['Eggs'] = [support = 0.6 , confidence = 1.0, lift =1.25] , dependent = True
9 - ['Kidney Beans', 'Eggs'] -> ['Onion'] = [support = 0.6 , confidence = 0.7499999999999999, lift =1.2499999999999998] , dependent = True
10 - ['Eggs', 'Onion'] -> 

In [41]:
for i, row in rules.iterrows():
    if row.lift > 1:
        print(f"{i+1} - {list(row.antecedents)} -> {list(row.consequents)} = [support = {row.support} , confidence = {row.confidence}, lift = {row.lift}] , dependent = {row.lift != 1.0}")

3 - ['Onion'] -> ['Eggs'] = [support = 0.6 , confidence = 1.0, lift = 1.25] , dependent = True
4 - ['Eggs'] -> ['Onion'] = [support = 0.6 , confidence = 0.7499999999999999, lift = 1.2499999999999998] , dependent = True
8 - ['Kidney Beans', 'Onion'] -> ['Eggs'] = [support = 0.6 , confidence = 1.0, lift = 1.25] , dependent = True
9 - ['Kidney Beans', 'Eggs'] -> ['Onion'] = [support = 0.6 , confidence = 0.7499999999999999, lift = 1.2499999999999998] , dependent = True
11 - ['Onion'] -> ['Kidney Beans', 'Eggs'] = [support = 0.6 , confidence = 1.0, lift = 1.25] , dependent = True
12 - ['Eggs'] -> ['Kidney Beans', 'Onion'] = [support = 0.6 , confidence = 0.7499999999999999, lift = 1.2499999999999998] , dependent = True


In [42]:
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
0,(Kidney Beans),(Eggs),1.0,0.8,0.8,0.80,1.00,1.0,0.00,1.0,0.0,0.80,0.000,0.900,1
1,(Eggs),(Kidney Beans),0.8,1.0,0.8,1.00,1.00,1.0,0.00,inf,0.0,0.80,0.000,0.900,1
2,(Onion),(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875,1
3,(Eggs),(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,1
4,(Milk),(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800,1
5,(Onion),(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800,1
6,(Yogurt),(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800,1
7,"(Kidney Beans, Onion)",(Eggs),0.6,0.8,0.6,1.00,1.25,1.0,0.12,inf,0.5,0.75,1.000,0.875,2
8,"(Kidney Beans, Eggs)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,2
9,"(Eggs, Onion)",(Kidney Beans),0.6,1.0,0.6,1.00,1.00,1.0,0.00,inf,0.0,0.60,0.000,0.800,2


- if you just need the associaltion rule of some antecedents or consequents

In [43]:
rules[rules['antecedents'] == {'Eggs', 'Kidney Beans'}]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
8,"(Kidney Beans, Eggs)",(Onion),0.8,0.6,0.6,0.75,1.25,1.0,0.12,1.6,1.0,0.75,0.375,0.875,2


if you need to query something by rules just do it :

- rules
    - support >= 0.7
    - confidence >= 0.9
    - lenth of items as anticedents = 1

In [44]:
rules[(rules['support'] >= 0.7) &
                    (rules['confidence'] >=0.8)&
                    (rules['antecedent_len'] ==1) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len
0,(Kidney Beans),(Eggs),1.0,0.8,0.8,0.8,1.0,1.0,0.0,1.0,0.0,0.8,0.0,0.9,1
1,(Eggs),(Kidney Beans),0.8,1.0,0.8,1.0,1.0,1.0,0.0,inf,0.0,0.8,0.0,0.9,1


## Another Example for Apriori: 

In [45]:
Transactions=  [['l1', 'l2', 'l5'],
                ['l2', 'l4'],
                ['l2', 'l3'],
                ['l1', 'l2', 'l4'],
                ['l1', 'l3'],
                ['l2', 'l3'],
                ['l1', 'l3'],
                ['l1', 'l2', 'l3', 'l5'],
                ['l1', 'l2', 'l3'],
                ['l1', 'l2']]

te = TransactionEncoder()
transactions_ar = te.fit(Transactions).transform(Transactions)
TrDataframe = pd.DataFrame(transactions_ar ,  columns= te.columns_)
TrDataframe

,l1,l2,l3,l4,l5
0,True,True,False,False,True
1,False,True,False,True,False
2,False,True,True,False,False
3,True,True,False,True,False
4,True,False,True,False,False
5,False,True,True,False,False
6,True,False,True,False,False
7,True,True,True,False,True
8,True,True,True,False,False
9,True,True,False,False,False


In [46]:
frequentSet =apriori(TrDataframe, min_support= 0.2 , use_colnames=True)
frequentSet

,support,itemsets
0,0.7,(l1)
1,0.8,(l2)
2,0.6,(l3)
3,0.2,(l4)
4,0.2,(l5)
5,0.5,"(l2, l1)"
6,0.4,"(l1, l3)"
7,0.2,"(l5, l1)"
8,0.4,"(l2, l3)"
9,0.2,"(l2, l4)"


![](https://github.com/AhmedKhalil777/DataScience.Learning/blob/master/Extracting%20Frequent%20pattern/min_confSlide.PNG?raw=1)

In [47]:
lecrules =association_rules(frequentSet, metric="confidence", min_threshold=0.7)
lecrules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(l1),(l2),0.7,0.8,0.5,0.714286,0.892857,1.0,-0.06,0.7,-0.285714,0.500000,-0.428571,0.669643
1,(l5),(l1),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
2,(l4),(l2),0.2,0.8,0.2,1.000000,1.250000,1.0,0.04,inf,0.250000,0.250000,1.000000,0.625000
3,(l5),(l2),0.2,0.8,0.2,1.000000,1.250000,1.0,0.04,inf,0.250000,0.250000,1.000000,0.625000
4,"(l1, l5)",(l2),0.2,0.8,0.2,1.000000,1.250000,1.0,0.04,inf,0.250000,0.250000,1.000000,0.625000
5,"(l2, l5)",(l1),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
6,(l5),"(l2, l1)",0.2,0.5,0.2,1.000000,2.000000,1.0,0.10,inf,0.625000,0.400000,1.000000,0.700000


# Frequent Itemsets via the FP-Growth Algorithm

In [48]:
from mlxtend.frequent_patterns import fpgrowth

fpgrowth(df, min_support=0.6, use_colnames=True)

,support,itemsets
0,1.0,(Kidney Beans)
1,0.8,(Eggs)
2,0.6,(Yogurt)
3,0.6,(Onion)
4,0.6,(Milk)
5,0.8,"(Kidney Beans, Eggs)"
6,0.6,"(Yogurt, Kidney Beans)"
7,0.6,"(Eggs, Onion)"
8,0.6,"(Kidney Beans, Onion)"
9,0.6,"(Eggs, Onion, Kidney Beans)"


# Compare execution time of Apriori and FP-Growth algorithms

In [49]:
%timeit -n 100 -r 10 apriori(df, min_support=0.6)

431 μs ± 42.5 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)


- %timeit measures how fast the algorithms execute.
- -n 100 runs the algorithm 100 times per experiment.
- -r 10 repeats the experiment 10 times for reliable results.

In [50]:
%timeit -n 100 -r 10 fpgrowth(df, min_support=0.6)

192 μs ± 9.05 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)


- Apriori uses candidate generation and is usually slower for large datasets.
- FP-Growth uses an FP-Tree structure and is generally faster and more efficient.

| Feature        | Apriori              | FP-Growth         |
| -------------- | -------------------- | ----------------- |
| Technique      | Candidate Generation | FP-Tree Structure |
| Database Scans | Multiple             | Fewer             |
| Memory Usage   | Higher               | Lower             |
| Speed          | Slower on large data | Faster            |
| Efficiency     | Less efficient       | More efficient    |
